# UnsloshをつかったLlama3.18bのSFT

### ライブラリロード

In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096 # 200ページの内容を効率よくパッキングするために4096〜8192を推奨
dtype = torch.bfloat16 # DGX SparkならBF16が最適
load_in_4bit = False   # 128GBメモリなら量子化なし(False)で最高精度を狙えます

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B", # または Qwen2.5-7B など
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 学習用のLoRA設定を追加（専門知識を効率よく注入するため）
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # 知識注入(CPT)の場合は少し大きめの数値を推奨
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"], # 全部入り
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260322. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████| 4/4 [01:39<00:00, 24.98s/it]


Unsloth: Offloading input_embeddings to disk to save VRAM


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
Unsloth 2026.3.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM


### PEFT/LoRAの設定

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # ランク数。知識注入には16〜32程度が推奨
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", # 知識を蓄えるMLP層
        "embed_tokens", "lm_head",           # 語彙と出力の層
    ],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Already have LoRA adapters! We shall skip this step.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


ステップ2：作成したJSONLデータの読み込み

In [10]:
from datasets import load_dataset

# ファイルパスを指定して読み込み
dataset = load_dataset("json", data_files={"train": "cpt_input.jsonl"}, split="train")

# データの形式を確認（"text" カラムがあることを想定）
print(dataset[0])

{'text': '# 障害程度別対象事業一覧表 (○対象・△一部対象)\n\n以下は、障害程度別の対象となる事業の一覧です。\n\n- **東京メトロ旅客運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **私鉄旅客運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **タクシー運賃の割引**（本文頁：120）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **航空運賃の割引**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **旅客船運賃の割引**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **テレビ受信料の減免**（本文頁：121）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが一部対象となります。所得制限があります。\n- **水道・下水道料金の減免**（本文頁：122）詳細は本文を参照してください。所得制限があります。\n- **携帯電話使用料等の割引**（本文頁：123）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありません。\n- **郵便料金・ゆうパック運賃等の減免**（本文頁：123）詳細は本文を参照してください。所得制限はありません。\n- **通常郵便葉書（青い鳥郵便葉書）の無料配布**（本文頁：124）視覚障害等級 1、2、3 および聴覚又は平衡感覚機能障害等級 2 が対象となります。所得制限はありません。\n\n### **税の軽減**\n\n税の軽減に関する事業は以下の通りです。\n\n- **所得税・住民税の障害者控除**（本文頁：125）視覚障害等級 1 から 6、聴覚又は平衡感覚機能障害等級 2 から 4 のすべてが対象となります。所得制限はありま

### 学習

In [11]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bf16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True, # これにより、1ページ1行のデータを隙間なく埋めて学習効率を最大化します
    args = TrainingArguments(
        per_device_train_batch_size = 4, # DGX Sparkならもっと増やせますが、まずは4から
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # まずは短く回して動作確認
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 学習開始！
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 218 | Num Epochs = 5 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 613,457,920 of 8,643,719,168 (7.10% trained)


Step,Training Loss
1,0.188900
2,0.166700
3,0.188800
4,0.223600
5,0.187500
6,0.190600
7,0.204600
8,0.171400
9,0.215600
10,0.212900


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


### 推論テスト

In [12]:
FastLanguageModel.for_inference(model) # 推論モードに切り替え
inputs = tokenizer(
    ["電車に関連する手続きは？"], 
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128)
print(tokenizer.batch_decode(outputs))


['<|begin_of_text|>電車に関連する手続きは？東京メトロ旅客運賃の割引については、所得制限はなく、身体障害者手帳の視覚障害（1〜6級）および聴覚又は平衡感覚機能障害（2〜4級）のすべての方が対象となります。詳細は本文頁 120 に記載されています。\n\n私鉄旅客運賃の割引についても、所得制限はなく、身体障害者手帳の視覚障害（1〜6級）および聴覚又は平衡感覚']
